# Parallel YouTube Match + Download Pipeline

**Goal:** Match YouTube videos AND download audio files with parallelization

**Speed:** ~3.6 seconds per song (with 5 workers)

**Expected:** ~24,000 songs per 24 hours

---

## Features

✅ **Parallel processing** (5 workers default, configurable)

✅ **Combined pipeline** (Match → Download in one operation)

✅ **Resume capability** (skip already processed songs)

✅ **Progress tracking** (saves after each batch)

✅ **Easy dataset splitting** (set START/END range for team collaboration)

✅ **Error handling** (one failure doesn't stop the whole batch)

## 1. Setup & Imports

In [33]:
import pandas as pd
import numpy as np
import re
import json
from pathlib import Path
from tqdm.auto import tqdm
import subprocess
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

## 2. Configuration

In [ ]:
# ====================
# DATASET SPLIT CONFIG
# ====================
# Set these to split work with your groupmate
START_INDEX = 0        # Start at song 0 (you)
END_INDEX = 57000      # End at song 57,000 (you)
# Groupmate would use: START_INDEX = 57000, END_INDEX = 114000

# Or set to None to process all songs
# START_INDEX = None
# END_INDEX = None

# ====================
# PERFORMANCE CONFIG
# ====================
NUM_WORKERS = 8  # Number of parallel workers (3-5 recommended)
BATCH_SIZE = 100  # Save progress every N songs

# ====================
# PATHS
# ====================
SPOTIFY_PATH = "../data/raw/spotify_tracks.csv"
AUDIO_OUTPUT_DIR = "../data/raw/youtube_audio/"
OUTPUT_PATH = "../data/raw/combined_match_download_results.csv"

# Create audio directory
Path(AUDIO_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"Dataset range: Songs {START_INDEX} to {END_INDEX}")
print(f"Workers: {NUM_WORKERS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Output: {OUTPUT_PATH}")

Dataset range: Songs 0 to 57000
Workers: 8
Batch size: 100
Output: ../data/raw/combined_match_download_results.csv


## 3. Load Spotify Dataset

In [36]:
# Load Spotify dataset
spotify_df = pd.read_csv(SPOTIFY_PATH)

print(f"Total songs in dataset: {len(spotify_df):,}")

# Apply range filter if specified
if START_INDEX is not None and END_INDEX is not None:
    spotify_df = spotify_df.iloc[START_INDEX:END_INDEX].copy()
    print(f"Filtered to range: {len(spotify_df):,} songs")

spotify_df.head()

Total songs in dataset: 114,000
Filtered to range: 57,000 songs


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


## 4. Core Functions

In [37]:
def normalize_text(text):
    """Normalize text for matching"""
    if pd.isna(text):
        return ""
    
    text = str(text).lower()
    text = re.sub(r"\(.*?\)", " ", text)
    text = re.sub(r"\[.*?\]", " ", text)
    
    noise_words = [
        "official", "video", "audio", "lyrics", "lyric",
        "music", "hd", "hq", "mv", "visualizer", "visualiser",
        "explicit", "clean", "version"
    ]
    for word in noise_words:
        text = re.sub(rf"\b{word}\b", " ", text)
    
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [38]:
def calculate_match_score(track_name, artist_name, youtube_title, youtube_channel):
    """Calculate match confidence with channel awareness and content type preferences"""
    track_norm = normalize_text(track_name)
    artist_norm = normalize_text(artist_name)
    title_norm = normalize_text(youtube_title)
    channel_norm = normalize_text(youtube_channel)
    
    title_lower = youtube_title.lower()
    channel_lower = youtube_channel.lower()
    
    # Preferred content
    preferred_content = ['official audio', 'official video', 'official music video']
    has_preferred = any(keyword in title_lower for keyword in preferred_content)
    
    # Secondary content
    secondary_content = ['lyric video', 'lyrics', 'lyric', 'visualizer']
    has_secondary = any(keyword in title_lower for keyword in secondary_content)
    
    # Variations (bad)
    variation_keywords = [
        'acoustic', 'remix', 'live', 'cover', 'karaoke',
        'instrumental', 'acapella', 'stripped', 'demo',
        'slowed', 'sped up', 'reverb', 'bass boosted'
    ]
    has_variation = any(keyword in title_lower for keyword in variation_keywords)
    
    # Label indicators
    label_indicators = [
        'vevo', 'records', 'music', 'entertainment', 'official',
        'topic', 'label', 'studio'
    ]
    has_label_indicator = any(indicator in channel_lower for indicator in label_indicators)
    
    # Token matching
    track_tokens = set(track_norm.split())
    artist_tokens = set(artist_norm.split())
    title_tokens = set(title_norm.split())
    channel_tokens = set(channel_norm.split())
    
    track_matches = len(track_tokens & title_tokens)
    artist_in_title = len(artist_tokens & title_tokens)
    artist_in_channel = len(artist_tokens & channel_tokens)
    
    is_likely_official = (artist_in_channel > 0) or has_label_indicator
    
    # Base scores
    track_score = track_matches / max(len(track_tokens), 1)
    artist_title_score = artist_in_title / max(len(artist_tokens), 1)
    artist_channel_score = artist_in_channel / max(len(artist_tokens), 1)
    
    # Weighted combination
    confidence = (track_score * 0.4) + (artist_title_score * 0.2) + (artist_channel_score * 0.4)
    
    # Apply modifiers
    if has_preferred:
        confidence = min(confidence * 1.2, 1.0)
    elif has_secondary:
        confidence = confidence * 0.9
    
    if has_variation:
        confidence = confidence * 0.5
    
    if not is_likely_official:
        confidence = confidence * 0.85
    
    return confidence

In [39]:
def search_youtube_ytdlp(track_name, artist_name, max_results=5, prefer_video=True):
    """Search YouTube using yt-dlp"""
    if prefer_video:
        query = f"{track_name} {artist_name} official music video"
    else:
        query = f"{track_name} {artist_name} official audio"
    
    try:
        cmd = [
            "yt-dlp",
            f"ytsearch{max_results}:{query}",
            "--dump-json",
            "--skip-download",
            "--no-warnings",
            "--quiet"
        ]
        
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=30
        )
        
        if result.returncode != 0:
            return []
        
        results = []
        for line in result.stdout.strip().split('\n'):
            if line.strip():
                try:
                    video_data = json.loads(line)
                    results.append({
                        "video_id": video_data.get("id", ""),
                        "title": video_data.get("title", ""),
                        "channel": video_data.get("uploader", ""),
                        "duration": video_data.get("duration", 0),
                        "view_count": video_data.get("view_count", 0)
                    })
                except json.JSONDecodeError:
                    continue
        
        return results
        
    except subprocess.TimeoutExpired:
        return []
    except Exception as e:
        return []

In [40]:
def find_best_match(track_name, artist_name, max_results=5):
    """Find best YouTube match with channel-aware scoring"""
    all_results = []
    
    # Search both video and audio
    video_results = search_youtube_ytdlp(track_name, artist_name, max_results, prefer_video=True)
    if video_results:
        for result in video_results:
            result['search_type'] = 'music_video'
        all_results.extend(video_results)
    
    audio_results = search_youtube_ytdlp(track_name, artist_name, max_results, prefer_video=False)
    if audio_results:
        for result in audio_results:
            result['search_type'] = 'official_audio'
        all_results.extend(audio_results)
    
    if not all_results:
        return None
    
    # Score all results
    scored_results = []
    for result in all_results:
        score = calculate_match_score(
            track_name, 
            artist_name, 
            result["title"],
            result["channel"]
        )
        scored_results.append({
            **result,
            "confidence": score
        })
    
    scored_results.sort(key=lambda x: x["confidence"], reverse=True)
    best = scored_results[0]
    
    return {
        "youtube_video_id": best["video_id"],
        "youtube_url": f"https://www.youtube.com/watch?v={best['video_id']}",
        "youtube_title": best["title"],
        "youtube_channel": best["channel"],
        "match_confidence": round(best["confidence"], 3),
        "content_type": best["search_type"]
    }

In [ ]:
def download_audio(youtube_url, output_path, track_id):
    """Download audio from YouTube as MP3 (much smaller!)"""
    try:
        # Change .wav to .mp3
        output_path = output_path.replace('.wav', '.mp3')
        output_template = output_path.replace('.mp3', '')
        
        cmd = [
            'yt-dlp',
            '--extract-audio',
            '--audio-format', 'mp3',  # ← Changed from wav
            '--audio-quality', '128K',  # 128 kbps MP3 = good quality, small size
            '--output', f'{output_template}.%(ext)s',
            '--no-playlist',
            '--no-warnings',
            '--quiet',
            '--no-progress',
            '--format', 'bestaudio',
            youtube_url
        ]
        
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=300
        )
        
        if Path(output_path).exists():
            file_size_mb = Path(output_path).stat().st_size / (1024 * 1024)
            
            # Safety check
            if file_size_mb > 10:  # MP3 shouldn't be >10 MB
                Path(output_path).unlink()
                return {
                    'status': 'failed',
                    'file_path': '',
                    'file_size_mb': 0,
                    'error_message': f'File too large ({file_size_mb:.1f} MB)'
                }
            
            return {
                'status': 'success',
                'file_path': output_path,
                'file_size_mb': round(file_size_mb, 2),
                'error_message': ''
            }
        else:
            return {
                'status': 'failed',
                'file_path': '',
                'file_size_mb': 0,
                'error_message': 'File not created'
            }
            
    except subprocess.TimeoutExpired:
        return {
            'status': 'failed',
            'file_path': '',
            'file_size_mb': 0,
            'error_message': 'Download timeout'
        }
    except Exception as e:
        return {
            'status': 'failed',
            'file_path': '',
            'file_size_mb': 0,
            'error_message': str(e)
        }

In [ ]:
def process_single_song(row):
    """Process one song: Match → Download"""
    track_id = row['track_id']
    track_name = row['track_name']
    artists = row['artists']
    
    result = {
        'track_id': track_id,
        'track_name': track_name,
        'artists': artists,
        'album_name': row.get('album_name', ''),
        'popularity': row.get('popularity', 0)
    }
    
    # Step 1: Match YouTube video
    match = find_best_match(track_name, artists)
    
    if not match:
        result.update({
            'match_found': False,
            'download_success': False,
            'youtube_video_id': '',
            'youtube_url': '',
            'youtube_title': '',
            'youtube_channel': '',
            'match_confidence': 0.0,
            'content_type': '',
            'file_path': '',
            'file_size_mb': 0,
            'error_message': 'No YouTube match found'
        })
        return result
    
    result['match_found'] = True
    result.update(match)
    
    # Step 2: Download audio
    output_path = f"{AUDIO_OUTPUT_DIR}{track_id}.mp3"
    
    # Skip if already exists
    if Path(output_path).exists():
        file_size_mb = Path(output_path).stat().st_size / (1024 * 1024)
        result.update({
            'download_success': True,
            'file_path': output_path,
            'file_size_mb': round(file_size_mb, 2),
            'error_message': 'Already downloaded (skipped)'
        })
        return result
    
    download_result = download_audio(match['youtube_url'], output_path, track_id)
    
    result.update({
        'download_success': download_result['status'] == 'success',
        'file_path': download_result['file_path'],
        'file_size_mb': download_result['file_size_mb'],
        'error_message': download_result['error_message']
    })
    
    return result

## 5. Check Existing Progress

In [43]:
output_path = Path(OUTPUT_PATH)

if output_path.exists():
    print("Found existing results file")
    existing_results = pd.read_csv(OUTPUT_PATH)
    print(f"Already processed: {len(existing_results):,} songs")
    print(f"  Matches found: {existing_results['match_found'].sum():,}")
    print(f"  Downloads successful: {existing_results['download_success'].sum():,}")
    
    processed_ids = set(existing_results['track_id'].values)
    pending_df = spotify_df[~spotify_df['track_id'].isin(processed_ids)].copy()
    print(f"\nRemaining: {len(pending_df):,} songs")
else:
    print("No existing progress. Starting fresh.")
    existing_results = None
    pending_df = spotify_df.copy()
    print(f"Total to process: {len(pending_df):,} songs")

No existing progress. Starting fresh.
Total to process: 57,000 songs


## 6. Parallel Processing

In [44]:
def process_batch_parallel(df, num_workers=5, batch_size=100):
    """
    Process songs in parallel with progress saving
    """
    all_results = []
    total_songs = len(df)
    
    # Load existing results
    if Path(OUTPUT_PATH).exists():
        existing = pd.read_csv(OUTPUT_PATH)
        all_results = existing.to_dict('records')
        print(f"Loaded {len(all_results):,} existing results\n")
    
    # Process in batches
    for batch_start in range(0, total_songs, batch_size):
        batch_end = min(batch_start + batch_size, total_songs)
        batch_df = df.iloc[batch_start:batch_end]
        
        print(f"\n{'='*70}")
        print(f"Batch: {batch_start + 1}-{batch_end} of {total_songs:,}")
        print(f"Workers: {num_workers}")
        print(f"{'='*70}\n")
        
        batch_results = []
        
        # Process batch in parallel
        with ThreadPoolExecutor(max_workers=num_workers) as executor:
            # Submit all tasks
            future_to_row = {
                executor.submit(process_single_song, row): idx 
                for idx, row in batch_df.iterrows()
            }
            
            # Collect results with progress bar
            with tqdm(total=len(batch_df), desc=f"Batch {batch_start//batch_size + 1}") as pbar:
                for future in as_completed(future_to_row):
                    try:
                        result = future.result()
                        batch_results.append(result)
                    except Exception as e:
                        # Handle worker failure
                        idx = future_to_row[future]
                        row = batch_df.loc[idx]
                        batch_results.append({
                            'track_id': row['track_id'],
                            'track_name': row['track_name'],
                            'artists': row['artists'],
                            'match_found': False,
                            'download_success': False,
                            'error_message': f'Worker error: {str(e)}'
                        })
                    finally:
                        pbar.update(1)
        
        # Add batch to all results
        all_results.extend(batch_results)
        
        # Save progress
        results_df = pd.DataFrame(all_results)
        results_df.to_csv(OUTPUT_PATH, index=False)
        
        # Show batch stats
        batch_matches = sum(r.get('match_found', False) for r in batch_results)
        batch_downloads = sum(r.get('download_success', False) for r in batch_results)
        
        print(f"\n✓ Batch complete!")
        print(f"  Total progress: {len(all_results):,} songs")
        print(f"  This batch - Matches: {batch_matches}/{len(batch_results)}")
        print(f"  This batch - Downloads: {batch_downloads}/{len(batch_results)}")
    
    return pd.DataFrame(all_results)

## 7. Run Processing

**⚠️ This will run for hours/days depending on dataset size!**

In [46]:
# Uncomment to run:

results = process_batch_parallel(
     pending_df, 
     num_workers=NUM_WORKERS, 
    batch_size=BATCH_SIZE
)


Batch: 1-100 of 57,000
Workers: 8



Batch 1:   0%|          | 0/100 [00:00<?, ?it/s]


✓ Batch complete!
  Total progress: 100 songs
  This batch - Matches: 99/100
  This batch - Downloads: 99/100

Batch: 101-200 of 57,000
Workers: 8



Batch 2:   0%|          | 0/100 [00:00<?, ?it/s]


✓ Batch complete!
  Total progress: 200 songs
  This batch - Matches: 100/100
  This batch - Downloads: 100/100

Batch: 201-300 of 57,000
Workers: 8



...


✓ Batch complete!
  Total progress: 5,100 songs
  This batch - Matches: 99/100
  This batch - Downloads: 99/100

Batch: 5101-5200 of 57,000
Workers: 8



Batch 52:   0%|          | 0/100 [00:00<?, ?it/s]


✓ Batch complete!
  Total progress: 5,200 songs
  This batch - Matches: 100/100
  This batch - Downloads: 100/100

Batch: 5201-5300 of 57,000
Workers: 8



Batch 53:   0%|          | 0/100 [00:00<?, ?it/s]

OSError: [Errno 28] No space left on device

## 8. Analyze Results

In [ ]:
if Path(OUTPUT_PATH).exists():
    results_df = pd.read_csv(OUTPUT_PATH)
    
    print("=== Combined Pipeline Results ===")
    print(f"\nTotal processed: {len(results_df):,}")
    print(f"\n--- Matching ---")
    print(f"Matches found: {results_df['match_found'].sum():,} ({results_df['match_found'].mean():.1%})")
    print(f"No match: {(~results_df['match_found']).sum():,}")
    
    matched = results_df[results_df['match_found']]
    if len(matched) > 0:
        print(f"\nAverage confidence: {matched['match_confidence'].mean():.3f}")
        print(f"High confidence (>0.8): {(matched['match_confidence'] > 0.8).sum():,}")
    
    print(f"\n--- Downloads ---")
    print(f"Downloads successful: {results_df['download_success'].sum():,} ({results_df['download_success'].mean():.1%})")
    print(f"Download failed: {(~results_df['download_success']).sum():,}")
    
    downloaded = results_df[results_df['download_success']]
    if len(downloaded) > 0:
        print(f"\nTotal audio size: {downloaded['file_size_mb'].sum():.2f} MB")
        print(f"Average file size: {downloaded['file_size_mb'].mean():.2f} MB")
    
    print(f"\n--- Content Types ---")
    if 'content_type' in matched.columns:
        print(matched['content_type'].value_counts())
else:
    print("No results yet. Run the processing cell first.")

## 9. Next Steps

✓ **You now have:** Audio files downloaded and ready for feature extraction

**Next:** Run parallelized feature extraction on the downloaded files (separate notebook)